In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")
secret_value_1 = user_secrets.get_secret("kaggle_key")
secret_value_2 = user_secrets.get_secret("KAGGLE_USERNAME")


In [ ]:
%cd /kaggle/working

!rm -rf /kaggle/working/ComfyUI

!git clone https://github.com/comfyanonymous/ComfyUI.git /kaggle/working/ComfyUI

%cd /kaggle/working/ComfyUI

!python -m pip install -q --upgrade pip
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q -r requirements.txt

print("ComfyUI installed")

In [ ]:
%cd /kaggle/working/ComfyUI

# New ComfyUI Manager is enabled from core with manager_requirements + --enable-manager
!test -f manager_requirements.txt && pip install -q -r manager_requirements.txt || echo "manager_requirements.txt not found"

print("Manager dependencies installed")

In [ ]:
%cd /kaggle/working/ComfyUI/custom_nodes

!rm -rf /kaggle/working/ComfyUI/custom_nodes/ComfyUI-GGUF
!git clone https://github.com/city96/ComfyUI-GGUF.git /kaggle/working/ComfyUI/custom_nodes/ComfyUI-GGUF

!pip install -q --upgrade gguf

!test -f /kaggle/working/ComfyUI/custom_nodes/ComfyUI-GGUF/requirements.txt && \
pip install -q -r /kaggle/working/ComfyUI/custom_nodes/ComfyUI-GGUF/requirements.txt || \
echo "No extra GGUF requirements file"

print("GGUF node installed")
print("Put GGUF UNet files in: /kaggle/working/ComfyUI/models/unet")

In [ ]:
%cd /kaggle/working/ComfyUI
!python main.py --listen 0.0.0.0 --port 8188 --enable-manager --lowvram --preview-method none

In [ ]:
!cd /kaggle/working/

In [ ]:
import subprocess


# Change 'zrok2_2.0.2' to 'zrok_2.0.2' in the filename part of the URL
url = "https://github.com/openziti/zrok/releases/download/v2.0.2/zrok_2.0.2_linux_amd64.tar.gz"

print("Downloading:", url)
# Note: I kept your local output name as 'zrok2.tar.gz' since that's what you likely want
subprocess.run(["wget", "-O", "zrok2.tar.gz", url], check=True)
subprocess.run(["tar", "-xzf", "zrok2.tar.gz"], check=True)

In [ ]:
!ls -lah /kaggle/working/zrok2

In [ ]:
!chmod +x /kaggle/working/zrok2
!/kaggle/working/zrok2 version

In [3]:
!/kaggle/working/zrok2 enable -v  R5nMRv9R2zgn

[ERROR] you already have an enabled environment, zrok2 disable first before you zrok2 enable


In [ ]:
import subprocess
import threading
import time
import os

# --- 1. CLEANUP ---
# Kill any old zrok processes that might be hanging in the background
!pkill -9 zrok2

# --- 2. CONFIGURATION ---
zrok_path = "/kaggle/working/zrok2"
comfy_path = "/kaggle/working/ComfyUI"

# --- 3. START ZROK (With Real-time Printing) ---
def start_zrok():
    global public_url
    # We use -u or PYTHONUNBUFFERED=1 to ensure we see logs immediately
    process = subprocess.Popen(
        [zrok_path, "share", "public", "http://localhost:8188", "--headless"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, # Capture errors too
        text=True,
        bufsize=1
    )
    
    print("🛰️  Zrok Log Feed Started:")
    for line in iter(process.stdout.readline, ''):
        clean_line = line.strip()
        if clean_line:
            print(f"  [zrok] {clean_line}") # This prints every line zrok generates
        
        if "access your share at" in line:
            public_url = line.split("at ")[1].strip()
            print("\n" + "⭐"*15)
            print(f"🚀 PUBLIC URL FOUND: {public_url}")
            print("⭐"*15 + "\n")

public_url = None
thread = threading.Thread(target=start_zrok, daemon=True)
thread.start()

# Wait up to 30 seconds for the URL
print("⏳ Waiting for zrok to handshake...")
for _ in range(30):
    if public_url: break
    time.sleep(1)

if not public_url:
    print("⚠️  Warning: URL not caught by script yet, but check the logs above!")

# --- 4. START COMFYUI ---
print(f"📦 Launching ComfyUI...")
os.chdir(comfy_path)
# Using your specific flags
!python main.py --listen 0.0.0.0 --port 8188 --enable-manager --lowvram --preview-method none

⏳ Waiting for zrok to handshake...
🛰️  Zrok Log Feed Started:
  [zrok] {"time":"2026-04-26T09:54:59.410644211Z","level":"INFO","source":{"function":"main.(*sharePublicCommand).shareLocal","file":"/__w/zrok/zrok/cmd/zrok2/sharePublic.go","line":281},"msg":"access your zrok share at the following endpoints:\n 2lzyz0v2ssk7.shares.zrok.io"}
⚠️  Warning: URL not caught by script yet, but check the logs above!
📦 Launching ComfyUI...
setup plugin alembic.autogenerate.schemas
setup plugin alembic.autogenerate.tables
setup plugin alembic.autogenerate.types
setup plugin alembic.autogenerate.constraints
setup plugin alembic.autogenerate.defaults
setup plugin alembic.autogenerate.comments
[START] Security scan
[ComfyUI-Manager] Using uv as Python module for pip operations.
Using Python 3.12.12 environment at: /usr
[DONE] Security scan
** ComfyUI startup time: 2026-04-26 09:55:24.935
** Platform: Linux
** Python version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
** Python executable: /usr/

In [ ]:
from kaggle_secrets import UserSecretsClient
secret_label = "your-secret-label"
secret_value = UserSecretsClient().get_secret(secret_label)

In [ ]:
%%bash
mkdir -p /kaggle/working/ComfyUI/models/unet
mkdir -p /kaggle/working/ComfyUI/models/text_encoders
mkdir -p /kaggle/working/ComfyUI/models/vae

wget --header="Authorization: Bearer $HF_TOKEN" \
  -O /kaggle/working/ComfyUI/models/unet/flux-2-klein-9b-kv-fp8.safetensors \
  "https://huggingface.co/black-forest-labs/FLUX.2-klein-9b-kv-fp8/resolve/main/flux-2-klein-9b-kv-fp8.safetensors"

wget --header="Authorization: Bearer $HF_TOKEN" \
  -O /kaggle/working/ComfyUI/models/text_encoders/qwen_3_8b_fp8mixed.safetensors \
  "https://huggingface.co/Comfy-Org/flux2-klein-9B/resolve/main/split_files/text_encoders/qwen_3_8b_fp8mixed.safetensors"

wget --header="Authorization: Bearer $HF_TOKEN" \
  -O /kaggle/working/ComfyUI/models/vae/flux2-vae.safetensors \
  "https://huggingface.co/Comfy-Org/flux2-dev/resolve/main/split_files/vae/flux2-vae.safetensors"

In [1]:
!ls -lh /kaggle/working/ComfyUI/models/unet/flux-2-klein-9b-kv-fp8.safetensors
!ls -lh /kaggle/working/ComfyUI/models/text_encoders/qwen_3_8b_fp8mixed.safetensors
!ls -lh /kaggle/working/ComfyUI/models/vae/flux2-vae.safetensors

-rw-r--r-- 1 root root 9.2G Apr 26 09:20 /kaggle/working/ComfyUI/models/unet/flux-2-klein-9b-kv-fp8.safetensors
-rw-r--r-- 1 root root 8.1G Apr 26 09:21 /kaggle/working/ComfyUI/models/text_encoders/qwen_3_8b_fp8mixed.safetensors
-rw-r--r-- 1 root root 321M Apr 26 09:21 /kaggle/working/ComfyUI/models/vae/flux2-vae.safetensors
